# 202 — Kmerseek performance vs AlphaFold2 pLDDT (genome-wide human-mouse orthologs)

**Hypothesis, from notebook 084.** Kmerseek reads sequence, not predicted structure, so its
ortholog-calling performance should be largely independent of AlphaFold2's per-protein confidence
(pLDDT). Notebook 084 tested this on ~9 QfO species with a single HP k=24 proxy encoding and a
sensitivity-to-first-false-positive metric. This notebook re-tests it at full genome scale on the
human→mouse ortholog task.

**Every combo with a real result on disk, discovered from the filesystem.** This notebook
originally checked notebook 200's protein/dayhoff top-10 plus 2 of the 6 HP variants over a narrow
k range, then migrated to `ou.load_all_alphabet_ksize_combos()` (notebook 200's own summary CSVs,
shared with 203/206/211). That CSV-based list has since gone stale relative to the raw pipeline
output -- the full ksize sweep filled in ~40 more combos (e.g. hp k20/21/23/25,
hp-kyte-doolittle k19-27, hp-thomas-dill(-no-c) k18-25) that the CSVs were never re-generated to
include. `ALL_COMBOS` is now built by scanning `DATA_DIR` (+ `ou.EXTRA_DATA_DIRS`) directly for
every `human_vs_mouse.{encoding}.k{ksize}.results.*` file that isn't empty/truncated, so it can't
drift from what's actually been run again.

**Ground truth:** MGI human-mouse ortholog pairs. **Performance metric:** per-human-gene
reciprocal-best-hit (RBH) correctness, the scoring rule notebook 200 validated against OrthoFinder
(see its section 2). **pLDDT:** bulk AlphaFold DB proteome downloads for human (UP000005640) and
mouse (UP000000589), mean pLDDT per canonical isoform from CA-atom B-factors.

**Sequence-based control.** OrthoFinder (pairwise sequence similarity, not structure) is included as
a second line: if kmerseek's flatness with pLDDT reflects "any sequence method is pLDDT-blind"
rather than something specific to kmerseek, OrthoFinder should show the same flat trend. No
genome-wide FoldSeek run exists for this ortholog task (the same gap noted in notebook 084), so a
structure-based comparator is not available here.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy import stats

sys.path.insert(0, str(Path(".").resolve()))
import ortholog_analysis_utils as ou

pl.Config.set_tbl_rows(15)

DATA_DIR      = Path("/Users/olga/data/gencode/results-human-mouse-orthologs")
PERGENE_DIR   = DATA_DIR / "202_pergene_cache"
PERGENE_DIR.mkdir(exist_ok=True)
OF_DIR        = Path("/Users/olga/data/gencode/data-for-orthofinder/OrthoFinder/Results_Mar03")
AF2_TAR_DIR   = Path("/Users/olga/data/alphafold_structures/proteome_tars")
QFO_DIR       = Path("/Users/olga/data/quest-for-orthologs/QfO_release_2020_04_with_updated_UP000008143/Eukaryota")
REPO          = Path("..").resolve()
OUT_DIR       = REPO / "data"
PREFIX        = "202_"

# ALL alphabets and ksizes with a real genome-wide results file on disk, discovered directly from
# the filesystem rather than from ou.load_all_alphabet_ksize_combos() (notebook 200's
# 200_alphabet_ksize_matched_scope_comparison.csv / 200_hp_variants_full_sweep.csv). Those two CSVs
# are a stale snapshot of whatever notebook 200 last swept -- after the full re-run that filled in
# every remaining ksize (e.g. hp k20/21/23/25, hp-kyte-doolittle k19-27, hp-thomas-dill(-no-c)
# k18-25), they were missing ~40 combos that already have real raw results on disk. Scanning the
# filesystem directly (same "derive from disk, not a notebook-written CSV" rule as the Nextflow
# side of this project, see [[feedback_pipeline_never_reads_notebook_output]]) means this notebook
# can't go stale the same way again.
import re

KNOWN_DASH_ENCODINGS = ["protein", "dayhoff", "hp", "hp-kyte-doolittle", "hp-lehninger",
                         "hp-lehninger-plus-c", "hp-pbotc-1st-ed", "hp-thomas-dill", "hp-thomas-dill-no-c"]
RESULTS_FILE_RE = re.compile(
    r"^human_vs_mouse\.(" + "|".join(re.escape(e) for e in KNOWN_DASH_ENCODINGS)
    + r")\.k(\d+)\.results\.(?:parquet|csv\.zst|csv\.gz)$")

search_dirs = [DATA_DIR] + [d for d in ou.EXTRA_DATA_DIRS if d != DATA_DIR]
found_combos = set()
for d in search_dirs:
    if not d.exists():
        continue
    for f in d.iterdir():
        m = RESULTS_FILE_RE.match(f.name)
        if m and not ou._is_empty_results_file(f):
            found_combos.add((m.group(1), int(m.group(2))))

DASH_FOR_DISPLAY = {enc.replace("-", "_"): enc for enc in KNOWN_DASH_ENCODINGS}
ALL_COMBOS = sorted((dash_enc.replace("-", "_"), k) for dash_enc, k in found_combos)
n_stale = ou.load_all_alphabet_ksize_combos().height
print(f"{len(ALL_COMBOS)} combos across {len({e for e, _ in ALL_COMBOS})} alphabets discovered on disk "
      f"(ou.load_all_alphabet_ksize_combos(), notebook 200's stale CSVs, would only give {n_stale})")

## 1. Per-gene kmerseek RBH correctness (every alphabet/k-size combo notebook 200 validated)

In [ ]:
# MGI ground truth
mgi_raw = pl.read_csv(DATA_DIR / "HOM_MouseHumanSequence.rpt", separator="\t",
                       null_values=[""], infer_schema_length=10_000)
human_rows = (mgi_raw.filter(pl.col("NCBI Taxon ID") == 9606)
              .select(["DB Class Key", "Symbol"]).rename({"Symbol": "human_symbol"}))
mouse_rows = (mgi_raw.filter(pl.col("NCBI Taxon ID") == 10090)
              .select(["DB Class Key", "Symbol"]).rename({"Symbol": "mouse_symbol"}))
mgi_pairs = (human_rows.join(mouse_rows, on="DB Class Key")
             .with_columns([pl.col("human_symbol").str.to_uppercase().alias("human_gene"),
                            pl.col("mouse_symbol").str.to_uppercase().alias("mouse_gene")])
             .select(["human_gene", "mouse_gene"]).unique())
truth_lists = mgi_pairs.group_by("human_gene").agg(pl.col("mouse_gene").alias("true_mouse_genes"))
print(f"MGI ortholog pairs: {len(mgi_pairs):,}")

In [ ]:
combo_frames = []
for enc, k in ALL_COMBOS:
    cache = PERGENE_DIR / f"{enc}_k{k}_jaccard.parquet"
    if cache.exists():
        df = pl.read_parquet(cache)
    else:
        dash_enc = DASH_FOR_DISPLAY[enc]
        try:
            df = ou.per_gene_rbh_table(dash_enc, k, truth_lists, DATA_DIR)
        except FileNotFoundError:
            print(f"  MISSING: {enc} k={k}")
            continue
        except pl.exceptions.NoDataError:
            print(f"  EMPTY: {enc} k={k}")
            continue
        except pl.exceptions.ComputeError as e:
            # e.g. human_vs_mouse.hp.k21.results.csv.gz -- confirmed truncated/corrupt with
            # `gzip -t` (see notebook 200's imports cell); skip rather than crash the sweep.
            print(f"  CORRUPT FILE, skipping: {enc} k={k}  ({type(e).__name__}: {e})")
            continue
        df = df.with_columns(pl.lit(enc).alias("encoding"))  # display label, not dash_enc
        df.write_parquet(cache)
    combo_frames.append(df)
    n_correct = int(df.filter(pl.col("has_mgi_truth"))["is_correct"].sum())
    n_truth = int(df["has_mgi_truth"].sum())
    print(f"{enc:22s} k={k:2d}: n_genes={df.height:6,d}  recall={n_correct/n_truth:.4f}  (n_with_truth={n_truth:,})")

combo_df = pl.concat(combo_frames)
combo_df = combo_df.with_columns((pl.col("encoding") + "_k" + pl.col("ksize").cast(pl.Utf8)).alias("combo"))
print(f"\nCombined per-gene table: {combo_df.height:,} rows across {combo_df['combo'].n_unique()} combos")

## 2. Per-gene OrthoFinder correctness (sequence-similarity control)

In [ ]:
OF_TSV = (OF_DIR / "Orthologues/Orthologues_gencode.v49.pc_translations"
          / "gencode.v49.pc_translations__v__gencode.vM38.pc_translations.tsv")

def gene_from_protein_id(pid: str) -> str:
    parts = pid.split("|")
    return parts[-2] if len(parts) >= 2 else pid

of_raw = pl.read_csv(OF_TSV, separator="\t", null_values=["", "nan"])
h_col, m_col = "gencode.v49.pc_translations", "gencode.vM38.pc_translations"

of_records = []
for row in of_raw.iter_rows(named=True):
    for h_pid in (row[h_col] or "").split(","):
        h_pid = h_pid.strip()
        if not h_pid:
            continue
        for m_pid in (row[m_col] or "").split(","):
            m_pid = m_pid.strip()
            if not m_pid:
                continue
            of_records.append({"human_gene": gene_from_protein_id(h_pid).upper(),
                                "mouse_gene": gene_from_protein_id(m_pid).upper()})
of_pairs = pl.DataFrame(of_records).unique()

# OrthoFinder calls one-or-more mouse genes per human gene; "correct" = ANY called mouse gene is MGI-true
of_called = of_pairs.group_by("human_gene").agg(pl.col("mouse_gene").alias("of_called_mouse_genes"))
of_correctness = (
    of_called.join(truth_lists, on="human_gene", how="left")
    .with_columns([
        pl.col("true_mouse_genes").is_not_null().alias("has_mgi_truth"),
        pl.struct(["of_called_mouse_genes", "true_mouse_genes"]).map_elements(
            lambda s: s["true_mouse_genes"] is not None
            and any(m in s["true_mouse_genes"] for m in s["of_called_mouse_genes"]),
            return_dtype=pl.Boolean,
        ).alias("is_correct"),
    ])
    .select(["human_gene", "has_mgi_truth", "is_correct"])
)
n_truth = int(of_correctness["has_mgi_truth"].sum())
n_correct = int(of_correctness.filter(pl.col("has_mgi_truth"))["is_correct"].sum())
print(f"OrthoFinder: n_genes={of_correctness.height:,}  recall={n_correct/n_truth:.4f}  (n_with_truth={n_truth:,})")

## 3. AlphaFold2 pLDDT — bulk proteome download and per-gene canonical extraction

Downloaded the full AlphaFold DB per-proteome archives for human (`UP000005640_9606_HUMAN_v6.tar`,
~5.2 GB) and mouse (`UP000000589_10090_MOUSE_v6.tar`, ~3.8 GB) rather than ~40,000 individual
per-accession requests (notebook 084's approach, workable for ~9 species-query proteins but not for
two full proteomes). pLDDT is the CIF `_atom_site.B_iso_or_equiv` value on CA atoms, one per
residue. UniProt accession → gene symbol comes from the QfO `Gene_Name` idmapping records; when a
gene has multiple isoform accessions, the longest is taken as canonical.

In [ ]:
import gzip
import tarfile


def extract_accession_plddt(tar_path: Path, cache_path: Path) -> pl.DataFrame:
    if cache_path.exists():
        return pl.read_parquet(cache_path)
    rows = []
    n_seen = 0
    with tarfile.open(tar_path, "r:") as tf:
        for member in tf:
            if not member.name.endswith("-model_v6.cif.gz"):
                continue
            acc = member.name.split("-")[1]
            fobj = tf.extractfile(member)
            if fobj is None:
                continue
            text = gzip.decompress(fobj.read()).decode("utf-8")
            plddts = [float(line.split()[14]) for line in text.splitlines()
                      if line.startswith("ATOM") and line.split()[3] == "CA"]
            n_seen += 1
            if not plddts:
                continue
            n = len(plddts)
            rows.append((acc, n, sum(plddts) / n, min(plddts),
                         sum(1 for p in plddts if p < 50) / n,
                         sum(1 for p in plddts if p < 70) / n))
            if n_seen % 5000 == 0:
                print(f"  ...{n_seen} structures parsed from {tar_path.name}", flush=True)
    df = pl.DataFrame(rows, schema=["accession", "n_residues", "mean_plddt", "min_plddt",
                                     "frac_plddt_lt50", "frac_plddt_lt70"], orient="row")
    df.write_parquet(cache_path)
    return df


def gene_name_map(idmapping_path: Path) -> pl.DataFrame:
    rows = []
    with open(idmapping_path) as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) == 3 and parts[1] == "Gene_Name":
                rows.append((parts[0], parts[2].upper()))
    return pl.DataFrame(rows, schema=["accession", "gene"], orient="row").unique()


def canonical_gene_plddt(accession_plddt: pl.DataFrame, gene_map: pl.DataFrame) -> pl.DataFrame:
    joined = accession_plddt.join(gene_map, on="accession", how="inner")
    return (joined.sort("n_residues", descending=True)
            .group_by("gene")
            .agg([pl.col("accession").first(), pl.col("n_residues").first(),
                  pl.col("mean_plddt").first(), pl.col("min_plddt").first(),
                  pl.col("frac_plddt_lt50").first(), pl.col("frac_plddt_lt70").first()]))

In [ ]:
HUMAN_TAR = AF2_TAR_DIR / "UP000005640_9606_HUMAN_v6.tar"
MOUSE_TAR = AF2_TAR_DIR / "UP000000589_10090_MOUSE_v6.tar"
HUMAN_IDMAP = QFO_DIR / "UP000005640_9606.idmapping"
MOUSE_IDMAP = QFO_DIR / "UP000000589_10090.idmapping"

human_acc_plddt = extract_accession_plddt(HUMAN_TAR, OUT_DIR / f"{PREFIX}human_accession_plddt.parquet")
mouse_acc_plddt = extract_accession_plddt(MOUSE_TAR, OUT_DIR / f"{PREFIX}mouse_accession_plddt.parquet")
print(f"human accessions with pLDDT: {human_acc_plddt.height:,}")
print(f"mouse accessions with pLDDT: {mouse_acc_plddt.height:,}")

human_gene_map = gene_name_map(HUMAN_IDMAP)
mouse_gene_map = gene_name_map(MOUSE_IDMAP)

human_plddt = canonical_gene_plddt(human_acc_plddt, human_gene_map)
mouse_plddt = canonical_gene_plddt(mouse_acc_plddt, mouse_gene_map)
human_plddt.write_parquet(OUT_DIR / f"{PREFIX}human_gene_plddt.parquet")
mouse_plddt.write_parquet(OUT_DIR / f"{PREFIX}mouse_gene_plddt.parquet")
print(f"human genes with canonical pLDDT: {human_plddt.height:,}")
print(f"mouse genes with canonical pLDDT: {mouse_plddt.height:,}")

## 3b. pLDDT distribution across the full human and mouse proteomes

What fraction of each proteome falls in the low-pLDDT bins used throughout this notebook. Every
canonical-isoform gene with a resolved AlphaFold2 structure (`human_plddt`/`mouse_plddt` from
section 3), not filtered to MGI-true orthologs and not restricted to any one alphabet/k-size combo.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(human_plddt["mean_plddt"], bins=50, range=(0, 100), alpha=0.55, color="#3062ac", label=f"human (n={human_plddt.height:,})")
ax.hist(mouse_plddt["mean_plddt"], bins=50, range=(0, 100), alpha=0.55, color="#c0392b", label=f"mouse (n={mouse_plddt.height:,})")
ax.axvline(70, color="#444", ls="--", lw=1, label="this notebook's low/high split (70)")
ax.set_xlabel("Mean pLDDT (canonical isoform)")
ax.set_ylabel("# genes")
ax.set_title("pLDDT distribution, full canonical proteome")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / f"{PREFIX}plddt_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

plddt_bins = [0, 50, 60, 70, 80, 90, 100]
plddt_bin_labels = ["<50", "50-60", "60-70", "70-80", "80-90", "90-100"]
for name, df in [("human", human_plddt), ("mouse", mouse_plddt)]:
    p = df["mean_plddt"]
    print(f"{name}: n={df.height:,}  mean={p.mean():.1f}  median={p.median():.1f}  std={p.std():.1f}")
    for lo, hi, lbl in zip(plddt_bins[:-1], plddt_bins[1:], plddt_bin_labels):
        n = df.filter((pl.col("mean_plddt") >= lo) & (pl.col("mean_plddt") < hi)).height
        print(f"    {lbl:8s}: {n:6,d}  ({n / df.height:.1%})")

## 4. Merge: per-gene kmerseek/OrthoFinder correctness + pLDDT

For each human gene with an MGI-documented true ortholog, attach the human protein's own pLDDT
(query side) and the true mouse ortholog's pLDDT (target side), using `min(human, mouse)` as the
pair's structural-confidence bottleneck, since either side being poorly predicted could degrade a
structure-based search. Rows without a canonical pLDDT on both sides are dropped, and the
denominator is reported.

In [ ]:
mouse_plddt_true = mouse_plddt.rename({"gene": "mouse_gene", "mean_plddt": "mouse_mean_plddt",
                                        "n_residues": "mouse_n_residues"}).select(
    ["mouse_gene", "mouse_mean_plddt", "mouse_n_residues"])
human_plddt_q = human_plddt.rename({"gene": "human_gene", "mean_plddt": "human_mean_plddt",
                                     "n_residues": "human_n_residues"}).select(
    ["human_gene", "human_mean_plddt", "human_n_residues"])

# true ortholog per human gene (first if multiple MGI-listed mouse orthologs) for target-side pLDDT
true_ortholog_one = mgi_pairs.group_by("human_gene").agg(pl.col("mouse_gene").first().alias("true_mouse_gene"))

analysis = (
    combo_df.filter(pl.col("has_mgi_truth"))
    .join(human_plddt_q, on="human_gene", how="left")
    .join(true_ortholog_one, on="human_gene", how="left")
    .join(mouse_plddt_true.rename({"mouse_gene": "true_mouse_gene"}), on="true_mouse_gene", how="left")
    .with_columns(
        pl.min_horizontal(["human_mean_plddt", "mouse_mean_plddt"]).alias("pair_min_plddt")
    )
)

n_total = analysis.height
n_with_plddt = int(analysis["pair_min_plddt"].is_not_null().sum())
print(f"Rows with MGI truth across all combos: {n_total:,}")
print(f"Rows with both human+mouse canonical pLDDT resolved: {n_with_plddt:,} "
      f"({n_with_plddt/n_total:.1%})")

analysis = analysis.filter(pl.col("pair_min_plddt").is_not_null())

## 5. Coverage check

Confirms the analyzable denominator (has MGI truth, and resolved pLDDT on both sides) does not
collapse at low pLDDT for any combo. If it did, a flat correctness-vs-pLDDT trend would be an
artifact of an empty low-pLDDT bin rather than evidence of pLDDT independence.

In [ ]:
bins = [0, 50, 60, 70, 80, 90, 100]
bin_labels = ["<50", "50-60", "60-70", "70-80", "80-90", "90-100"]

# Different k-sizes cover different human-gene universes (higher k = fewer hits for some genes,
# per notebook 200's n_genes_scope), so coverage is checked PER combo, not assumed uniform.
cov_rows = []
for enc, k in ALL_COMBOS:
    label = f"{enc}_k{k}"
    sub = analysis.filter(pl.col("combo") == label)
    row = {"combo": label}
    for lo, hi, lbl in zip(bins[:-1], bins[1:], bin_labels):
        row[lbl] = sub.filter((pl.col("pair_min_plddt") >= lo) & (pl.col("pair_min_plddt") < hi)).height
    cov_rows.append(row)
cov_df = pl.DataFrame(cov_rows)
print("Coverage (n genes with MGI truth + resolved pLDDT) by pLDDT bin, per combo:")
print(cov_df)
min_bin_n = min(cov_df[lbl].min() for lbl in bin_labels)
if min_bin_n < 20:
    print(f"\n  At least one (combo, bin) cell has < 20 genes (min={min_bin_n}) -- "
          "treat that cell's contribution to the trend as suggestive only.")

## 6. Primary analysis: RBH/OrthoFinder correctness vs pLDDT, across all combos

In [ ]:
# Point-biserial correlation (is_correct binary vs continuous pLDDT) per combo
corr_rows = []
for combo in ALL_COMBOS:
    label = f"{combo[0]}_k{combo[1]}"
    sub = analysis.filter(pl.col("combo") == label)
    if sub.height < 30:
        continue
    r, p = stats.pointbiserialr(sub["is_correct"].to_numpy().astype(int), sub["pair_min_plddt"].to_numpy())
    corr_rows.append({"combo": label, "encoding": combo[0], "ksize": combo[1], "n": sub.height,
                       "point_biserial_r": r, "p_value": p})

of_sub = of_correctness.filter(pl.col("has_mgi_truth")).join(
    analysis.filter(pl.col("combo") == f"{ALL_COMBOS[0][0]}_k{ALL_COMBOS[0][1]}")
    .select(["human_gene", "pair_min_plddt"]), on="human_gene", how="inner")
r_of, p_of = stats.pointbiserialr(of_sub["is_correct"].to_numpy().astype(int),
                                    of_sub["pair_min_plddt"].to_numpy())
corr_rows.append({"combo": "OrthoFinder", "encoding": "OrthoFinder", "ksize": 0, "n": of_sub.height,
                   "point_biserial_r": r_of, "p_value": p_of})

corr_df = pl.DataFrame(corr_rows).sort("point_biserial_r")
corr_df

## 7. Figure

In [ ]:
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8,
    "axes.linewidth": 0.6,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

# Heatmap (alphabet x ksize) instead of one line-overlay panel per alphabet family -- ~100 combos
# breaks a per-combo-line layout the same way section 3's per-combo figures in notebook 200 did
# (see [[all_alphabets_ksizes_utility]]'s scale caveat); the old 4-key CMAPS dict this used to use
# would KeyError outright once there are 9 alphabets instead of 3 (+hp_thomas_dill).
non_of = corr_df.filter(pl.col("encoding") != "OrthoFinder")
alpha_order = (non_of.group_by("encoding").agg(pl.col("point_biserial_r").abs().mean().alias("mean_abs_r"))
               .sort("mean_abs_r", descending=True)["encoding"].to_list())
ksize_order = sorted(non_of["ksize"].unique().to_list())

pivot = non_of.pivot(index="encoding", on="ksize", values="point_biserial_r").to_pandas().set_index("encoding")
# polars .pivot() column labels are always strings ("26", not 26) even though the source column
# is Int64 -- reindexing straight against the int ksize_order silently matched nothing and left
# every cell NaN (the empty left panel). Cast columns back to int before reindexing.
pivot.columns = pivot.columns.astype(int)
pivot = pivot.reindex(index=alpha_order, columns=ksize_order)

fig, (axHM, axB) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1.3, 1]})

r_max = float(non_of["point_biserial_r"].abs().max())
im = axHM.imshow(pivot.to_numpy(), aspect="auto", cmap="RdBu_r", vmin=-r_max, vmax=r_max)
axHM.set_yticks(range(len(alpha_order)))
axHM.set_yticklabels(alpha_order, fontsize=7)
axHM.set_xticks(range(len(ksize_order)))
axHM.set_xticklabels(ksize_order, fontsize=7)
axHM.set_xlabel("k-size")
axHM.set_title("Point-biserial r (is_correct vs pair min pLDDT)\nby alphabet x k-size")
fig.colorbar(im, ax=axHM, label="r", shrink=0.85)

# Mean |r| per alphabet, OrthoFinder shown as a reference line (not a bar -- it's one number, not
# an alphabet family)
mean_abs_r = (non_of.group_by("encoding").agg(pl.col("point_biserial_r").abs().mean().alias("mean_abs_r"))
              .sort("mean_abs_r"))
of_row = corr_df.filter(pl.col("encoding") == "OrthoFinder").row(0, named=True)

axB.barh(mean_abs_r["encoding"], mean_abs_r["mean_abs_r"], color="#3062ac")
axB.axvline(abs(of_row["point_biserial_r"]), color="#7f7f7f", ls="--", lw=1.5,
            label=f"OrthoFinder |r|={abs(of_row['point_biserial_r']):.3f}")
axB.set_xlabel("Mean |point-biserial r| across this alphabet's k-sizes")
axB.set_title("Which alphabets show a real pLDDT correlation?")
axB.legend(fontsize=7, loc="lower right")
axB.grid(True, alpha=0.25, axis="x")

plt.tight_layout()
plt.savefig(OUT_DIR / f"{PREFIX}kmerseek_vs_plddt.png", dpi=150, bbox_inches="tight")
plt.show()

## 7b. XY view: RBH-correctness rate vs pLDDT, per alphabet

The heatmap and bar above compress each combo to a single point-biserial *r*, which ranks alphabets
but hides the shape of the relationship. This plots it directly: for each alphabet, the fraction of
MGI-true human genes called correctly, binned by pair-min pLDDT (the same bins as section 5),
pooled across that alphabet's k-sizes (bold line, shaded 95% Wilson CI), with each k-size traced
faintly underneath so a pooled trend driven by one k shows up as diverging faint lines. The
OrthoFinder control is overlaid as a dashed reference in every panel. Bins with fewer than 20 genes
are dropped, matching section 5.

In [ ]:
from statsmodels.stats.proportion import proportion_confint


def binned_accuracy(is_correct: np.ndarray, plddt: np.ndarray, min_n: int = 20) -> pl.DataFrame:
    rows = []
    for lo, hi, lbl in zip(bins[:-1], bins[1:], bin_labels):
        mask = (plddt >= lo) & (plddt < hi)
        n = int(mask.sum())
        if n < min_n:
            continue
        n_correct = int(is_correct[mask].sum())
        lo_ci, hi_ci = proportion_confint(n_correct, n, method="wilson")
        rows.append({"bin": lbl, "mid": (lo + hi) / 2, "acc": n_correct / n, "n": n,
                      "lo_ci": lo_ci, "hi_ci": hi_ci})
    return pl.DataFrame(rows, schema={"bin": pl.Utf8, "mid": pl.Float64, "acc": pl.Float64,
                                       "n": pl.Int64, "lo_ci": pl.Float64, "hi_ci": pl.Float64})


of_binned = binned_accuracy(of_sub["is_correct"].to_numpy().astype(int), of_sub["pair_min_plddt"].to_numpy())

n_alpha = len(alpha_order)
ncols = 3
nrows = -(-n_alpha // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), sharex=True, sharey=True)
axes = axes.flatten()
cmap = plt.get_cmap("viridis")

for ax, enc in zip(axes, alpha_order):
    ksizes_here = sorted(non_of.filter(pl.col("encoding") == enc)["ksize"].unique().to_list())
    k_min, k_max = min(ksizes_here), max(ksizes_here)
    for k in ksizes_here:
        sub = analysis.filter(pl.col("combo") == f"{enc}_k{k}")
        b = binned_accuracy(sub["is_correct"].to_numpy().astype(int), sub["pair_min_plddt"].to_numpy())
        if b.height == 0:
            continue
        color = cmap((k - k_min) / max(k_max - k_min, 1))
        ax.plot(b["mid"], b["acc"], color=color, lw=0.7, alpha=0.5)

    pooled = analysis.filter(pl.col("encoding") == enc)
    pb = binned_accuracy(pooled["is_correct"].to_numpy().astype(int), pooled["pair_min_plddt"].to_numpy())
    ax.plot(pb["mid"], pb["acc"], color="#c0392b", lw=2, marker="o", ms=3, label="pooled (all k)")
    ax.fill_between(pb["mid"], pb["lo_ci"], pb["hi_ci"], color="#c0392b", alpha=0.15)

    ax.plot(of_binned["mid"], of_binned["acc"], color="#7f7f7f", lw=1.2, ls="--", label="OrthoFinder")

    r_row = mean_abs_r.filter(pl.col("encoding") == enc).row(0, named=True)
    ax.set_title(f"{enc}\nmean |r|={r_row['mean_abs_r']:.3f}, k={k_min}-{k_max}", fontsize=8)
    ax.set_ylim(0, 1.02)
    ax.grid(True, alpha=0.2)

for ax in axes[n_alpha:]:
    ax.axis("off")
axes[0].legend(fontsize=6, loc="lower right")

bottom_row_start = max(n_alpha - ncols, 0)
for ax in axes[bottom_row_start:n_alpha]:
    ax.set_xlabel("pair min pLDDT (bin midpoint)")
for i in range(0, n_alpha, ncols):
    axes[i].set_ylabel("Fraction RBH-correct")

plt.tight_layout()
plt.savefig(OUT_DIR / f"{PREFIX}accuracy_vs_plddt_xy_by_alphabet.png", dpi=150, bbox_inches="tight")
plt.show()

## 7c. Is OrthoFinder's line in 7b flat?

Section 7b's OrthoFinder reference line looks flat next to the kmerseek curves, but flat next to a
38-point swing is not the same as flat. This checks it directly: OrthoFinder's binned accuracy with
95% Wilson CIs, so low-n bins (including the smallest pLDDT bin) show their real width, plus a
chi-square test of whether accuracy differs across bins at all.

In [ ]:
from scipy.stats import chi2_contingency

of_ic = of_sub["is_correct"].to_numpy().astype(int)
of_pl = of_sub["pair_min_plddt"].to_numpy()

of_bin_rows = []
for lo, hi, lbl in zip(bins[:-1], bins[1:], bin_labels):
    mask = (of_pl >= lo) & (of_pl < hi)
    n = int(mask.sum())
    n_correct = int(of_ic[mask].sum())
    lo_ci, hi_ci = proportion_confint(n_correct, n, method="wilson")
    of_bin_rows.append({"bin": lbl, "mid": (lo + hi) / 2, "n": n, "n_correct": n_correct,
                         "acc": n_correct / n, "lo_ci": lo_ci, "hi_ci": hi_ci})
of_bin_df = pl.DataFrame(of_bin_rows)
print(f"OrthoFinder, n={of_sub.height:,} MGI-true human genes with resolved pLDDT:")
print(of_bin_df)

chi2, chi2_p, dof, _ = chi2_contingency(
    [of_bin_df["n_correct"].to_list(), (of_bin_df["n"] - of_bin_df["n_correct"]).to_list()])
pooled_acc = of_bin_df["n_correct"].sum() / of_bin_df["n"].sum()
acc_range = of_bin_df["acc"].max() - of_bin_df["acc"].min()
print(f"\nPooled accuracy: {pooled_acc:.4f}")
print(f"Range across bins: {of_bin_df['acc'].min():.4f} - {of_bin_df['acc'].max():.4f} ({acc_range:.4f}, "
      f"i.e. {acc_range * 100:.1f} percentage points)")
print(f"Chi-square test (accuracy independent of pLDDT bin): chi2={chi2:.2f}, dof={dof}, p={chi2_p:.2e}")
if chi2_p < 0.05:
    print(f"  -> statistically NOT flat -- n is large enough that even a {acc_range * 100:.1f}-point "
          "wiggle reaches significance, but see the plot below for whether that wiggle is practically meaningful")
else:
    print("  -> consistent with flat (no significant difference across pLDDT bins)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(of_bin_df["mid"], of_bin_df["acc"],
            yerr=[of_bin_df["acc"] - of_bin_df["lo_ci"], of_bin_df["hi_ci"] - of_bin_df["acc"]],
            fmt="o-", color="#7f7f7f", capsize=3, label="OrthoFinder")
ax.axhline(pooled_acc, color="#7f7f7f", ls=":", lw=1, alpha=0.6, label=f"pooled acc={pooled_acc:.3f}")
for _, row in enumerate(of_bin_df.iter_rows(named=True)):
    ax.annotate(f"n={row['n']:,}", (row["mid"], row["hi_ci"]), textcoords="offset points",
                xytext=(0, 4), ha="center", fontsize=6, color="#555")
ax.set_ylim(0.9, 1.02)
ax.set_xlabel("pair min pLDDT (bin midpoint)")
ax.set_ylabel("Fraction RBH-correct (OrthoFinder)")
ax.set_title(f"OrthoFinder accuracy vs pLDDT, with 95% CI\n"
             f"range={acc_range*100:.1f} pts, chi2 p={chi2_p:.1e}")
ax.legend(fontsize=8, loc="lower right")
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(OUT_DIR / f"{PREFIX}orthofinder_plddt_flatness_check.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Written readout

In [ ]:
print("=" * 70)
print("ANALYSIS READOUT -- genome-wide kmerseek RBH correctness vs AlphaFold2 pLDDT")
print("=" * 70)
print(f"\nCombos analyzed: {len(ALL_COMBOS)} across {len({e for e, _ in ALL_COMBOS})} alphabets "
      f"(protein, dayhoff, hp, and all 6 dash-named HP variants -- every ksize with a real "
      f"genome-wide results file on disk)")
print(f"Denominator: MGI-true human genes with resolved human+mouse canonical pLDDT")
print(f"  n = {n_with_plddt:,} (out of {n_total:,} MGI-true rows before the pLDDT join)")

print("\nPoint-biserial correlation (is_correct vs pair min pLDDT), sorted weakest-to-strongest |r|:")
for row in corr_df.sort(pl.col("point_biserial_r").abs()).iter_rows(named=True):
    sig = "**" if row["p_value"] < 0.01 else ("*" if row["p_value"] < 0.05 else "ns")
    print(f"  {row['combo']:22s}  r={row['point_biserial_r']:+.4f}  p={row['p_value']:.3e} {sig}  (n={row['n']:,})")

max_abs_r = float(corr_df["point_biserial_r"].abs().max())
print(f"\nLargest |r| across all combos + OrthoFinder control: {max_abs_r:.4f}")

print("\nMean |r| per alphabet (see Figure heatmap/bar above for the full breakdown):")
print(corr_df.filter(pl.col("encoding") != "OrthoFinder")
      .group_by("encoding").agg(pl.col("point_biserial_r").abs().mean().alias("mean_abs_r"))
      .sort("mean_abs_r", descending=True))

## 8b. Metric choice: does pLDDT change which composite score ranks candidate pairs best?

Section 9's finding concerns a *binary* call (RBH-correct or not) correlating weakly with pLDDT for
HP-family alphabets. A different question, following notebooks 200/205: among candidate (human,
mouse) pairs, does the best-performing *continuous* ranking score (of the same 12 composites)
differ between low- and high-pLDDT human genes? Notebook 203 found that disorder quartile flips the
winner — raw `enrichment` beats the p-value composite within quartiles even though the composite
wins pooled — and pLDDT and disorder are correlated.

**Every HP alphabet and k-size, not one combo.** This originally checked hp_thomas_dill k=26,
chosen because it had the strongest pLDDT correlation at the time. That is the same one-combo blind
spot section 1 called out for the correctness-vs-pLDDT analysis: a metric-winner flip could be
specific to thomas_dill's containment-saturation quirk (notebooks 200/205) rather than a property
of HP alphabets generally. This now sweeps all 7 HP alphabets across their full k range
(`HP_COMBOS`, filtered from `ALL_COMBOS`). Protein and dayhoff are excluded because sections 1/7
showed their pLDDT correlation is negligible. Split at pLDDT = 70, this notebook's own bin edge
from sections 5/6.

In [ ]:
mgi_pairs_ou, mgi_set = ou.load_mgi_orthologs()
plddt_map = dict(zip(human_plddt["gene"].to_list(), human_plddt["mean_plddt"].to_list()))

HP_COMBOS = [(enc, k) for enc, k in ALL_COMBOS if enc.startswith("hp")]
print(f"{len(HP_COMBOS)} HP-family combos across {len({enc for enc, _ in HP_COMBOS})} alphabets "
      "(was 1 hand-picked combo before this section was swept)")


def ortholog_evaluation_tsv(dash_enc: str, k: int) -> Path | None:
    # dash-named HP variants ship ortholog_evaluation.*.tsv.zst; plain "hp" ships .tsv.gz instead
    # (an older pipeline run predating the .zst conversion) -- try both rather than hardcoding one.
    # A handful of files are header-only stubs (e.g. hp k=22, 51 bytes) from a run that indexed
    # an empty database, same empty-index bug ou._is_empty_results_file guards against elsewhere;
    # skip those rather than crash pl.scan_csv on an empty CSV.
    for suffix in (".tsv.zst", ".tsv.gz"):
        f = DATA_DIR / f"ortholog_evaluation.{dash_enc}.k{k}{suffix}"
        if f.exists() and f.stat().st_size > 1024:
            return f
    return None


flip_rows = []
for enc, k in HP_COMBOS:
    dash_enc = DASH_FOR_DISPLAY[enc]
    tsv = ortholog_evaluation_tsv(dash_enc, k)
    if tsv is None:
        print(f"  MISSING/EMPTY tsv: {enc} k={k}")
        continue
    df, _, _, _ = ou.load_kmerseek_data(kmerseek_tsv=tsv, mgi_ortholog_set=mgi_set)
    df = df.with_columns(pl.col("human_gene").replace_strict(plddt_map, default=None).alias("human_plddt"))
    df = ou.add_composite_scores(df).with_columns(pl.col("is_mgi_ortholog").alias("label"))
    df = df.filter(pl.col("human_plddt").is_not_null())

    low = df.filter(pl.col("human_plddt") < 70)
    high = df.filter(pl.col("human_plddt") >= 70)
    if low.height < 30 or high.height < 30 or int(low["label"].sum()) < 5 or int(high["label"].sum()) < 5:
        print(f"  SKIP (too few labeled pairs in one pLDDT half): {enc} k={k}")
        continue

    low_winner = ou.leaderboard_df(ou.compute_aucs(low)).sort("rank").row(0, named=True)["metric"]
    high_winner = ou.leaderboard_df(ou.compute_aucs(high)).sort("rank").row(0, named=True)["metric"]
    flipped = low_winner != high_winner
    flip_rows.append({"encoding": enc, "ksize": k, "n_low": low.height, "n_high": high.height,
                       "winner_low_plddt": low_winner, "winner_high_plddt": high_winner, "flipped": flipped})
    print(f"{enc:22s} k={k:2d}: low-pLDDT winner={low_winner:22s} high-pLDDT winner={high_winner:22s}"
          f" {'FLIP' if flipped else ''}")

flip_df = pl.DataFrame(flip_rows)
n_flipped = int(flip_df["flipped"].sum())
print(f"\n{flip_df.height} HP combos checked ({len(HP_COMBOS) - flip_df.height} skipped for missing"
      f" data or too few labeled pairs in one pLDDT half)")
print(f"{n_flipped} / {flip_df.height} show a metric-winner flip between low- and high-pLDDT halves")
print("\nWinner counts, low-pLDDT half:")
print(flip_df["winner_low_plddt"].value_counts().sort("count", descending=True))
print("\nWinner counts, high-pLDDT half:")
print(flip_df["winner_high_plddt"].value_counts().sort("count", descending=True))
if n_flipped:
    print("\nCombos with a flip:")
    print(flip_df.filter(pl.col("flipped")))

# Figure: how often each metric wins, low- vs high-pLDDT half, across all 78 HP combos
low_counts = flip_df["winner_low_plddt"].value_counts().rename({"winner_low_plddt": "metric"})
high_counts = flip_df["winner_high_plddt"].value_counts().rename({"winner_high_plddt": "metric"})
all_metrics = sorted(set(low_counts["metric"].to_list()) | set(high_counts["metric"].to_list()))
low_map = dict(zip(low_counts["metric"].to_list(), low_counts["count"].to_list()))
high_map = dict(zip(high_counts["metric"].to_list(), high_counts["count"].to_list()))

fig, ax = plt.subplots(figsize=(7, max(2.5, 0.35 * len(all_metrics))))
y = np.arange(len(all_metrics))
h = 0.35
ax.barh(y + h / 2, [low_map.get(m, 0) for m in all_metrics], height=h, color="#3062ac", label="low pLDDT (<70) winner")
ax.barh(y - h / 2, [high_map.get(m, 0) for m in all_metrics], height=h, color="#c0392b", label="high pLDDT (≥70) winner")
ax.set_yticks(y)
ax.set_yticklabels(all_metrics, fontsize=8)
ax.set_xlabel(f"# of {flip_df.height} HP combos where this metric had rank 1")
ax.set_title("Best-ranking composite score, by pLDDT half -- all HP alphabets/k-sizes")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.25, axis="x")
plt.tight_layout()
plt.savefig(OUT_DIR / f"{PREFIX}metric_winner_counts_by_plddt.png", dpi=150, bbox_inches="tight")
plt.show()

**pLDDT flips the metric winner for most HP combos, reversing the single-combo check.** Of the 78
HP-family combos, 73 had enough labeled pairs in both pLDDT halves; 5 were skipped (`hp` k=22 and
the k=18/19 ends of `hp_kyte_doolittle`/`hp_lehninger_plus_c`/`hp_pbotc_1st_ed` have missing or
empty result files). **44 of those 73 (60%) show a different rank-1 metric between the low- and
high-pLDDT halves.** The combo originally checked here, `hp_thomas_dill` k=26, is one of the 29
that does not flip (jaccard wins both halves), which is why the old single-combo verdict reported
no flip. The same alphabet flips at k=19-21 and k=28-30, just not at k=22-27.

Which metric wins where follows a systematic pattern. `score_enr_cont` (continuous enrichment) is
the low-pLDDT winner in 33/73 combos vs 13/73 at high pLDDT, while `score_enr_cont_freq` (its
frequency-weighted variant) is almost exclusively a high-pLDDT phenomenon (28 wins at high vs 2 at
low). `jaccard` is the most pLDDT-stable of the three common winners (29 low, 32 high) but is still
not universal. This parallels notebook 203's disorder-quartile result: structural-confidence
stratification changes which composite ranks candidate pairs best for most HP combos, and it is not
visible from any single combo. Every raw-p-value-alone score (`score_neglogp`,
`score_bonf_neglogp`, `score_bh_neglogq`) is essentially absent from the winner counts in both
halves, consistent with notebooks 200/205's containment-saturation finding.

## 9. Verdict

**Old 14-combo finding, superseded, kept for reference.** The pLDDT-independence hypothesis held
for the finer alphabets but not for HP. Across the 10 top protein/dayhoff combos from notebook 200,
correctness-vs-pLDDT correlation stayed small and sign-inconsistent (-0.029 to +0.009), matching
the OrthoFinder control (+0.037). The two HP-family alphabets included at that point differed:
hp_thomas_dill k=25 (r=+0.072) and k=26 (r=+0.060), and to a lesser extent hp Lehninger k=26/27
(r=+0.017 to +0.021).

**Full 97-combo sweep (9 alphabets, sections 6/7): the HP-vs-finer-alphabet split holds and
sharpens.** Ranked by mean |point-biserial r| across each alphabet's full k range:
`hp_thomas_dill_no_c` 0.105, `hp_kyte_doolittle` 0.077, `hp_thomas_dill` 0.054, `hp_lehninger`
0.046, `hp_lehninger_plus_c` 0.043, `hp_pbotc_1st_ed` 0.039, `hp` 0.024, `dayhoff` 0.014, `protein`
0.013. The OrthoFinder control (r=0.037) sits between the two groups: below all 5 higher-effect HP
variants, above plain `hp`, `dayhoff`, and `protein`. The largest single effect in the sweep is
`hp_thomas_dill_no_c` k=26 (r=+0.166, n=17,822), more than 2x the OrthoFinder control and nearly 3x
the old run's largest HP effect. The heatmap also shows a few small negative cells
(`hp_thomas_dill` k=19-21, `hp_kyte_doolittle` k=22-27), but all are n=938, an order of magnitude
below the ~17,800-gene denominator elsewhere; notebook 200's imports cell flags several of these
low-k HP files as truncated runs, so the sign flip there is noise.

**Point-biserial r understates the effect: section 7b's binned-accuracy curves show pLDDT
independence holds for no kmerseek alphabet, only to different degrees.** Pooled across each
alphabet's full k range, fraction correct at the lowest pLDDT bin (<50) vs its peak bin (70-80):

| alphabet | acc at pLDDT<50 | acc at peak (70-80) | drop |
|---|---|---|---|
| protein | 0.831 | 0.940 | 11 pts |
| dayhoff | 0.798 | 0.916 | 12 pts |
| hp | 0.658 | 0.860 | 20 pts |
| hp_pbotc_1st_ed | 0.615 | 0.858 | 24 pts |
| hp_lehninger | 0.599 | 0.844 | 25 pts |
| hp_thomas_dill | 0.580 | 0.859 | 28 pts |
| hp_kyte_doolittle | 0.564 | 0.857 | 29 pts |
| hp_thomas_dill_no_c | 0.443 | 0.826 | **38 pts** |
| OrthoFinder | 0.985 | 0.995 (80-90 bin) | ~1 pt (flat) |

Every kmerseek alphabet, including the two whose point-biserial r reads as negligible, loses real
accuracy at low pLDDT. The r-based ranking compresses a 38-point swing and an 11-point swing into
numbers that look 8x apart. OrthoFinder is flat: 98-99% correct in every pLDDT bin, never below
97.8%. So the claim is not "protein/dayhoff are pLDDT-independent, HP is not" but "every kmerseek
alphabet pays a structure-quality-correlated accuracy penalty that OrthoFinder mostly does not, and
the size of that penalty scales with how much sequence information the alphabet discards" — protein
20 letters, dayhoff 6, HP 2, matching the order of the table.

**Mechanistic reading.** The 2-letter HP alphabet discards the most sequence information per
residue and depends most on long k-mers for specificity. Full-alphabet protein and 6-letter dayhoff
k-mers retain more discriminative signal and are hurt less, but are still hurt. Low pLDDT often
marks compositionally-biased or low-complexity/disordered regions, and collapsing to fewer letters
plausibly makes those regions more prone to spurious over- or under-matching as the alphabet
coarsens. This notebook measures the correlation and its accuracy scale, not the mechanism; see
[[lowcomplexity_null_pipeline]] for the null model built to test it.

**Practical implication.** For the project's best genome-wide alphabets (protein k=13-15, dayhoff
k=19-20, per [[genome_wide_alphabet_ksize_ranking]]), pLDDT is a real but modest confound of ~11-12
accuracy points from best to worst bin: worth a caveat on any recall number quoted without a pLDDT
breakdown, not a reason to change alphabet. For HP-family results the confound is 20-38 points, so
any HP-based recall/AUC number should be read as conditional on the query proteome's
structural-confidence mix. `hp_pbotc_1st_ed` (notebook 200's best HP variant genome-wide, per
[[genome_wide_alphabet_ksize_ranking]]) sits mid-pack on this axis — a 24-point drop, mean
|r|=0.039 — not the worst (`hp_thomas_dill_no_c`), and less pLDDT-robust than `hp` itself
(20-point drop, mean |r|=0.024, the least pLDDT-sensitive of the 7).